In [1]:
#!/usr/bin/env python3
"""Generate a balanced 6k command dataset from the intent source files.

Source layout in this folder:
- LearningNER.json     -> learning_create
- !LearningNER.json    -> add, delete, show, update, behavioral

For each intent, the script samples 1000 texts and writes one final
merged file named commandDataset.json with rows in this format:
{
  "text": "...",
  "intent": "command"
}
"""

import argparse
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional


TARGET_INTENTS = [
    "add",
    "delete",
    "show",
    "update",
    "behavioral",
    "learning_create",
    "learning_delete",
    "learning_update"
]

SOURCE_FILES = {
    "learning_delete": ["LearningNER.json"],
    "learning_update": ["LearningNER.json"],
    "learning_create": ["LearningNER.json"],
    "add": ["!LearningNER.json"],
    "delete": ["!LearningNER.json"],
    "show": ["!LearningNER.json"],
    "update": ["!LearningNER.json"],
    "behavioral": ["!LearningNER.json"],
}

COMMON_TEXT_KEYS = ("text", "Text", "sentence", "utterance", "content", "message")


def extract_text_from_item(item: Any) -> List[str]:
    texts = []

    if isinstance(item, str):
        cleaned = item.strip()
        return [cleaned] if cleaned else []

    if isinstance(item, dict):
        for key in COMMON_TEXT_KEYS:
            value = item.get(key)
            if isinstance(value, str) and value.strip():
                return [value.strip()]

        for value in item.values():
            if isinstance(value, str) and value.strip():
                texts.append(value.strip())

        for value in item.values():
            if isinstance(value, dict):
                for key in COMMON_TEXT_KEYS:
                    nested = value.get(key)
                    if isinstance(nested, str) and nested.strip():
                        return [nested.strip()]

    return texts


def extract_intent_from_item(item: Any) -> Optional[str]:
    if isinstance(item, dict):
        intent = item.get("intent")
        if isinstance(intent, str) and intent.strip():
            return intent.strip()
    return None


def load_texts(path: Path, intent_filter: Optional[str] = None) -> List[str]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON array in {path}")

    texts = []
    for item in data:
        if intent_filter is not None:
            item_intent = extract_intent_from_item(item)
            if item_intent is not None and item_intent != intent_filter:
                continue

        candidates = extract_text_from_item(item)
        if candidates:
            texts.append(candidates[0])

    seen = set()
    unique_texts = []
    for text in texts:
        if text not in seen:
            seen.add(text)
            unique_texts.append(text)

    return unique_texts


def generate_instances(texts: List[str], count: int) -> List[Dict[str, str]]:
    if not texts:
        return []
    return [{"text": random.choice(texts), "intent": "command"} for _ in range(count)]


def main():
    parser = argparse.ArgumentParser(allow_abbrev=False)
    parser.add_argument(
        "--folder",
        default=".",
        help="folder containing the source files; defaults to the notebook working directory",
    )
    parser.add_argument(
        "--per-intent",
        type=int,
        default=1000,
        help="number of command samples to generate for each intent",
    )
    args, _unknown = parser.parse_known_args()

    folder = Path(args.folder).resolve()
    per_intent = max(0, int(args.per_intent))

    all_instances = []

    for intent_name in TARGET_INTENTS:
        file_names = SOURCE_FILES.get(intent_name, [])
        if not file_names:
            print(f"Warning: no source files configured for intent: {intent_name}")
            continue

        collected_texts = []
        for file_name in file_names:
            source_path = folder / file_name
            texts = load_texts(source_path, intent_filter=intent_name)
            collected_texts.extend(texts)

        seen = set()
        unique_texts = []
        for text in collected_texts:
            if text not in seen:
                seen.add(text)
                unique_texts.append(text)

        print(f"{intent_name}: {len(unique_texts)} unique texts loaded")

        if not unique_texts:
            raise ValueError(f"No texts found for intent: {intent_name}")

        intent_instances = generate_instances(unique_texts, per_intent)
        all_instances.extend(intent_instances)
        print(f"{intent_name}: generated {len(intent_instances)} command instances")

    random.shuffle(all_instances)

    out_path = folder / "commandDataset.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(all_instances, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(all_instances)} instances to {out_path}")


if __name__ == "__main__":
    main()


add: 2500 unique texts loaded
add: generated 1000 command instances
delete: 2200 unique texts loaded
delete: generated 1000 command instances
show: 2079 unique texts loaded
show: generated 1000 command instances
update: 2767 unique texts loaded
update: generated 1000 command instances
behavioral: 154 unique texts loaded
behavioral: generated 1000 command instances
learning_create: 4594 unique texts loaded
learning_create: generated 1000 command instances
learning_delete: 1661 unique texts loaded
learning_delete: generated 1000 command instances
learning_update: 3672 unique texts loaded
learning_update: generated 1000 command instances
Wrote 8000 instances to /Users/hamidshehzad/FYP_CODE_20_March/FYP_CODE_INSHALLAH_FINAL --- VSCODE/DataSets/InshallahDataset/LatestProperDatasets/commandDataset.json
